# Conformal predictive systems for time series

This notebook covers discrete demand and continuous measurements with Nixtla-compatible panel forecasters. Both CPS variants calibrate series- and horizon-specific residual distributions and return a self-contained, panel-aligned forecast.

In [1]:
import os
import sys
sys.path.append(os.path.abspath("../.."))

import numpy as np
import pandas as pd
from IPython.display import display
from mlforecast import MLForecast
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

from tinyconformal.series import (
    ContinuousTimeSeriesConformalPredictiveSystem,
    DiscreteTimeSeriesConformalPredictiveSystem,
)
from tinyconformal.evaluation import PanelEvaluator, FirstStageEvaluator
from tinyconformal.utils import NewsvendorSolver

pd.set_option("display.max_columns", 20)

## 1. Discrete demand

The target is a non-negative integer count. The last seven days are held out so evaluation uses observations that were not used during fitting.

In [2]:
def make_count_panel(n_periods=140, seed=42):
    rng = np.random.default_rng(seed)
    dates = pd.date_range("2025-01-01", periods=n_periods, freq="D")
    frames = []
    for offset, unique_id in enumerate(["store_A", "store_B"]):
        mean = 4.0 + offset + np.linspace(0, 1.5, n_periods) + 1.5 * (dates.dayofweek >= 5)
        frames.append(pd.DataFrame({
            "unique_id": unique_id, "ds": dates, "y": rng.poisson(mean)
        }))
    return pd.concat(frames, ignore_index=True)

horizon = 7
count_data = make_count_panel()
count_train = count_data.groupby("unique_id", group_keys=False).head(-horizon)
count_test = count_data.groupby("unique_id", group_keys=False).tail(horizon)
count_data.head()

,unique_id,ds,y
0,store_A,2025-01-01,6
1,store_A,2025-01-02,3
2,store_A,2025-01-03,6
3,store_A,2025-01-04,9
4,store_A,2025-01-05,7


In [3]:
count_learner = MLForecast(
    models={"LinearRegression": LinearRegression()},
    freq="D", lags=[1, 7, 14], date_features=["dayofweek"],
)
count_cps = DiscreteTimeSeriesConformalPredictiveSystem(
    learner=count_learner,
    dispersion_learner=RandomForestRegressor(
        n_estimators=100, min_samples_leaf=3, random_state=42, n_jobs=-1
    ),
    minimum=0,
).fit(count_train, horizon=horizon, n_windows=5, static_features=[], n_jobs=1)
count_forecast = count_cps.predict_distribution(h=horizon)
count_forecast.to_frame().head()

,unique_id,ds,LinearRegression
0,store_A,2025-05-14,4.378222
1,store_A,2025-05-15,4.524704
2,store_A,2025-05-16,4.935391
3,store_A,2025-05-17,5.504575
4,store_A,2025-05-18,6.329747


### Discrete PPF, CDF, SF, and PMF

Each method returns a DataFrame on the same `unique_id`/`ds` grid. For stock level `N`, `cdf(N)` is the service probability `P(Y<=N)` and `sf(N)` is the exceedance risk `P(Y>N)`.

In [4]:
display(count_forecast.ppf([0.50, 0.90, 0.95]).head())
inventory_level = 8
service = count_forecast.cdf(inventory_level)
risk = count_forecast.sf(inventory_level)
probabilities = service.join(risk[[f"P(Y>{inventory_level})"]])
probabilities["probability_check"] = (
    probabilities[f"P(Y<={inventory_level})"] + probabilities[f"P(Y>{inventory_level})"]
)
display(probabilities.head())
count_forecast.pmf([0, 5, 8]).head()

,unique_id,ds,LinearRegression,Q(0.5),Q(0.9),Q(0.95)
0,store_A,2025-05-14,4.378222,3,7,7
1,store_A,2025-05-15,4.524704,2,7,7
2,store_A,2025-05-16,4.935391,2,14,14
3,store_A,2025-05-17,5.504575,6,10,10
4,store_A,2025-05-18,6.329747,8,14,14


,unique_id,ds,LinearRegression,P(Y<=8),P(Y>8),probability_check
0,store_A,2025-05-14,4.378222,1.000000,0.000000,1.0
1,store_A,2025-05-15,4.524704,1.000000,0.000000,1.0
2,store_A,2025-05-16,4.935391,0.666667,0.333333,1.0
3,store_A,2025-05-17,5.504575,0.666667,0.333333,1.0
4,store_A,2025-05-18,6.329747,0.500000,0.500000,1.0


,unique_id,ds,LinearRegression,P(Y=0),P(Y=5),P(Y=8)
0,store_A,2025-05-14,4.378222,0.166667,0.000000,0.000000
1,store_A,2025-05-15,4.524704,0.166667,0.000000,0.000000
2,store_A,2025-05-16,4.935391,0.000000,0.000000,0.000000
3,store_A,2025-05-17,5.504575,0.000000,0.166667,0.166667
4,store_A,2025-05-18,6.329747,0.166667,0.000000,0.166667


### Held-out evaluation and first-stage diagnostics

Both evaluations use the final seven days reserved as a test set. The predictive forecast evaluates complete central intervals, while `FirstStageEvaluator` summarizes point forecasts independently of conformal scaling.

In [5]:
count_test = count_test.sort_values(["unique_id", "ds"]).reset_index(drop=True)
count_observed = count_test["y"].to_numpy()
display(PanelEvaluator.evaluate_interval(
    count_test, forecast=count_forecast, coverages=[0.80, 0.90, 0.95]
))
display(PanelEvaluator.evaluate_distribution(
    count_test, forecast=count_forecast, train_df=count_train
))

count_first_stage_test = count_forecast.to_frame().merge(
    count_test[["unique_id", "ds", "y"]],
    on=["unique_id", "ds"],
    how="inner",
    validate="one_to_one",
)
display(FirstStageEvaluator.evaluate(
    count_first_stage_test, prediction_col="LinearRegression"
) )
FirstStageEvaluator.calibration_table(
    count_first_stage_test, prediction_col="LinearRegression", n_bins=5
)

,model,coverage,coverage_rate,interval_width_mean,mwis,n_obs
0,LinearRegression,0.80,0.786,8.0,11.571,14
1,LinearRegression,0.90,0.786,8.0,15.143,14
2,LinearRegression,0.95,0.786,8.0,22.286,14


,unique_id,crps,target_std,ncrps,n_obs
0,store_A,1.811363,2.530110,0.715923,7
1,store_B,2.230986,2.553997,0.873528,7


,wape,pbias,score,forecast_instability,false_demand_on_zero_days_avg_pred,peak_demand_deviation
0,0.3246,-0.1511,0.4758,0.1331,0.0,-0.1511


,calibration_bin,count,mean_prediction,mean_observed,mean_residual
0,"(4.377, 4.983]",3,4.612772,5.000000,0.387228
1,"(4.983, 5.445]",3,5.248078,6.333333,1.085255
2,"(5.445, 6.006]",2,5.547074,7.500000,1.952926
3,"(6.006, 6.36]",3,6.189625,5.000000,-1.189625
4,"(6.36, 7.351]",3,6.848871,10.000000,3.151129


### Optimize discrete inventory

A time-series CPS forecast can be passed directly to the solver. Its panel and underlying distribution are already aligned.

In [6]:
count_plan = NewsvendorSolver.optimize_distribution(
    count_forecast, underage_cost=9.0, overage_cost=1.0
)
display(count_plan.head())
NewsvendorSolver.marginal_benefit_distribution(
    count_forecast, underage_cost=9.0, overage_cost=1.0, units=range(0, 11, 2)
).head()

,unique_id,ds,LinearRegression,critical_ratio,y_optimal
0,store_A,2025-05-14,4.378222,0.9,7.0
1,store_A,2025-05-15,4.524704,0.9,7.0
2,store_A,2025-05-16,4.935391,0.9,14.0
3,store_A,2025-05-17,5.504575,0.9,10.0
4,store_A,2025-05-18,6.329747,0.9,14.0


,unique_id,ds,LinearRegression,MB(k=0),MB(k=2),MB(k=4),MB(k=6),MB(k=8),MB(k=10)
0,store_A,2025-05-14,4.378222,9.0,7.333333,4.000000,2.333333,-1.000000,-1.000000
1,store_A,2025-05-15,4.524704,9.0,5.666667,2.333333,2.333333,-1.000000,-1.000000
2,store_A,2025-05-16,4.935391,9.0,7.333333,4.000000,4.000000,2.333333,2.333333
3,store_A,2025-05-17,5.504575,9.0,7.333333,7.333333,5.666667,4.000000,2.333333
4,store_A,2025-05-18,6.329747,9.0,7.333333,7.333333,7.333333,5.666667,2.333333


## 2. Continuous measurements

Continuous CPS uses the same panel workflow, but retains real-valued support and therefore does not expose a PMF.

In [7]:
def make_continuous_panel(n_periods=140, seed=7):
    rng = np.random.default_rng(seed)
    dates = pd.date_range("2025-01-01", periods=n_periods, freq="D")
    frames = []
    for offset, unique_id in enumerate(["region_A", "region_B"]):
        t = np.arange(n_periods)
        y = (10 + 2 * offset + 0.02 * t + 1.5 * np.sin(2 * np.pi * t / 7)
             + rng.normal(0, 0.8 + 0.2 * offset, n_periods))
        frames.append(pd.DataFrame({"unique_id": unique_id, "ds": dates, "y": y}))
    return pd.concat(frames, ignore_index=True)

continuous_data = make_continuous_panel()
continuous_train = continuous_data.groupby("unique_id", group_keys=False).head(-horizon)
continuous_test = continuous_data.groupby("unique_id", group_keys=False).tail(horizon)

In [8]:
continuous_learner = MLForecast(
    models={"LinearRegression": LinearRegression()},
    freq="D", lags=[1, 7, 14], date_features=["dayofweek"],
)
continuous_cps = ContinuousTimeSeriesConformalPredictiveSystem(
    learner=continuous_learner,
    dispersion_learner=RandomForestRegressor(
        n_estimators=100, min_samples_leaf=3, random_state=43, n_jobs=-1
    ),
).fit(continuous_train, horizon=horizon, n_windows=10, static_features=[], n_jobs=1, step_size=horizon - 3)
continuous_forecast = continuous_cps.predict_distribution(h=horizon)

### Continuous PPF, CDF, SF, intervals, and decisions

The distribution interface is the same as in the discrete case, except that quantiles and optimal quantities remain continuous.

In [11]:
continuous_forecast.ppf([0.05, 0.50, 0.95]).head()

,unique_id,ds,LinearRegression,Q(0.05),Q(0.5),Q(0.95)
0,region_A,2025-05-14,11.763821,10.086159,11.866182,13.265461
1,region_A,2025-05-15,13.740267,11.701902,14.260318,15.773686
2,region_A,2025-05-16,13.172945,11.169518,13.197189,13.661697
3,region_A,2025-05-17,13.485322,11.439408,13.556117,14.432627
4,region_A,2025-05-18,11.480983,9.783170,11.699208,13.035224


In [12]:
continuous_forecast.cdf([10, 15]).head()

,unique_id,ds,LinearRegression,P(Y<=10),P(Y<=15)
0,region_A,2025-05-14,11.763821,0.000000,1.000000
1,region_A,2025-05-15,13.740267,0.000000,0.818182
2,region_A,2025-05-16,13.172945,0.000000,1.000000
3,region_A,2025-05-17,13.485322,0.000000,1.000000
4,region_A,2025-05-18,11.480983,0.181818,1.000000


In [13]:
continuous_forecast.sf([10, 15]).head()

,unique_id,ds,LinearRegression,P(Y>10),P(Y>15)
0,region_A,2025-05-14,11.763821,1.000000,0.000000
1,region_A,2025-05-15,13.740267,1.000000,0.181818
2,region_A,2025-05-16,13.172945,1.000000,0.000000
3,region_A,2025-05-17,13.485322,1.000000,0.000000
4,region_A,2025-05-18,11.480983,0.818182,0.000000


In [14]:
continuous_forecast.interval(coverage=0.90).head()

,unique_id,ds,LinearRegression,Q(0.05),Q(0.95)
0,region_A,2025-05-14,11.763821,10.086159,13.265461
1,region_A,2025-05-15,13.740267,11.701902,15.773686
2,region_A,2025-05-16,13.172945,11.169518,13.661697
3,region_A,2025-05-17,13.485322,11.439408,14.432627
4,region_A,2025-05-18,11.480983,9.783170,13.035224


In [15]:
PanelEvaluator.evaluate_interval(
    continuous_test, forecast=continuous_forecast, coverages=[0.80, 0.90, 0.95]
)

,model,coverage,coverage_rate,interval_width_mean,mwis,n_obs
0,LinearRegression,0.80,0.857,2.839,5.523,14
1,LinearRegression,0.90,0.929,4.075,6.437,14
2,LinearRegression,0.95,0.929,4.075,8.800,14


In [16]:
PanelEvaluator.evaluate_distribution(
    continuous_test, forecast=continuous_forecast, train_df=continuous_train
)

,unique_id,crps,target_std,ncrps,n_obs
0,region_A,0.599411,1.489583,0.402402,7
1,region_B,0.923382,1.608940,0.573907,7


In [17]:
NewsvendorSolver.optimize_distribution(
    continuous_forecast, underage_cost=6.0, overage_cost=2.0
).head()

,unique_id,ds,LinearRegression,critical_ratio,y_optimal
0,region_A,2025-05-14,11.763821,0.75,12.767810
1,region_A,2025-05-15,13.740267,0.75,14.814628
2,region_A,2025-05-16,13.172945,0.75,13.615194
3,region_A,2025-05-17,13.485322,0.75,14.299525
4,region_A,2025-05-18,11.480983,0.75,12.117160


## Support comparison

| Target | CDF | SF | PPF | PMF | Newsvendor output |
|---|---:|---:|---:|---:|---|
| Non-negative integer counts | Yes | Yes | Yes | Yes | Integer |
| Continuous values | Yes | Yes | Yes | No | Continuous |

Unlike the tabular cross-conformal CPS, time-series calibration is performed separately by series and forecast horizon using sequential rolling-origin windows.